# Experiment 2 — Home Credit Default Risk

Reproduces:
- **Table 3**: MSE / AUC / KS for GBM and QPM
  - GBM: AUC=0.7517, KS=0.3784, MSE=0.0683
  - QPM (quantized): AUC=0.7487, KS=0.3740, MSE=0.0683
- **Section 4.2**: Churn — QPM fine-tuning reduces churn by **3.4x** vs GBM retrain

**Dataset**: Home Credit Default Risk (Kaggle competition)
~300k accounts · 8.0% default rate · requires Kaggle API token

**Prerequisite** — Kaggle token setup (see `data/README.md`):
The notebook auto-downloads `application_train.csv` on first run.

**Installation** (run once from repo root):
```bash
pip install -e ..
```

In [ ]:
# ── CONFIGURATION — edit here, then Kernel → Restart & Run All ────────────────
# [1] Single-entry config block
FAST_RUN = False      # True → <3 min smoke-test on CPU
DEVICE   = "cpu"      # xgboost tree_method hint

# [6] All seeds declared explicitly
SEED_SPLIT = 42
SEED_GBM   = 857
SEED_BOOT  = 0

# Paths
import pathlib
ARTIFACTS  = pathlib.Path("../artifacts/exp02")
SPLITS_DIR = ARTIFACTS / "splits"
DATA_DIR   = pathlib.Path("../data/home_credit")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Hyper-parameters — [8] FAST_RUN overrides
N_ESTIMATORS     = 100  if FAST_RUN else 400
N_BOOT           = 200  if FAST_RUN else 2000
LEARNING_RATE    = 0.03
MAX_DEPTH        = 4
SUBSAMPLE        = 0.7
COLSAMPLE_BYTREE = 0.7
MIN_CHILD_WEIGHT = 10
GAMMA            = 2

print(f"FAST_RUN={FAST_RUN}  |  N_ESTIMATORS={N_ESTIMATORS}  |  N_BOOT={N_BOOT}")
print(f"SEED_SPLIT={SEED_SPLIT}  |  SEED_GBM={SEED_GBM}  |  SEED_BOOT={SEED_BOOT}")

In [ ]:
import os, sys, random, hashlib, json as _json, pathlib, importlib, subprocess, zipfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from qpm import (
    QPMQuantizer, AnchoredCalibration,
    auc_score, ks_score, mse_score, bootstrap_ci, delong_test, churn_metrics,
    WoEScorecard,
)


# [2] Determinism helper — call before every model fit
def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)


# [5] Environment lock — print all package versions
_PKGS = ["numpy", "pandas", "sklearn", "xgboost", "scipy", "matplotlib"]
print(f"Python {sys.version.split()[0]}")
for _p in _PKGS:
    try:
        _mod = importlib.import_module(_p)
        print(f"  {_p}: {_mod.__version__}")
    except Exception:
        print(f"  {_p}: NOT FOUND")

In [ ]:
# [2] Determinism probe — runs before any modelling; must pass
seed_everything(SEED_GBM)
_Xp = np.random.default_rng(0).random((200, 8))
_yp = (_Xp[:, 0] + 0.3 * _Xp[:, 1] > 0.7).astype(float)

def _quick_gbm(s):
    # subsample + colsample_bytree ensure the random_state actually drives
    # different tree structures, making the same/different-seed check reliable
    m = xgb.XGBRegressor(n_estimators=20, learning_rate=0.1,
                          subsample=0.7, colsample_bytree=0.7,
                          random_state=s, tree_method="hist")
    m.fit(_Xp, _yp)
    return m.predict(_Xp[:5])

_p1, _p2, _p3 = _quick_gbm(42), _quick_gbm(42), _quick_gbm(99)
assert np.allclose(_p1, _p2), "FAIL: same seed produced different predictions!"
assert not np.allclose(_p1, _p3), "FAIL: different seeds produced identical predictions!"
print("Determinism probe PASSED")
del _Xp, _yp, _p1, _p2, _p3

In [ ]:
# [3] Data-provenance hash utilities
def _sha256_df(df, nrows=None):
    sub = df if nrows is None else df.iloc[:nrows]
    return hashlib.sha256(sub.to_csv(index=False).encode()).hexdigest()


def _verify_hash(name, h, cache_path):
    cache = pathlib.Path(cache_path)
    known = _json.loads(cache.read_text()) if cache.exists() else {}
    if name in known:
        assert known[name] == h, (
            f"Data hash mismatch for '{name}'! "
            f"Expected {known[name][:16]}... got {h[:16]}... "
            "Delete artifacts/*/data_hashes.json to reset."
        )
        print(f"  {name}: hash OK")
    else:
        known[name] = h
        cache.write_text(_json.dumps(known, indent=2))
        print(f"  {name}: hash saved (first run) -- {h[:16]}...")


# [4] Deterministic splits — indices persisted to disk
def _get_or_create_splits(X, y, test_size, seed, tag, splits_dir):
    sd = pathlib.Path(splits_dir)
    tr_p = sd / f"{tag}_train.npy"
    te_p = sd / f"{tag}_test.npy"
    if tr_p.exists() and te_p.exists():
        tr_idx, te_idx = np.load(str(tr_p)), np.load(str(te_p))
        print(f"  Loaded split '{tag}' from disk")
    else:
        all_idx = np.arange(len(y))
        tr_idx, te_idx = train_test_split(
            all_idx, test_size=test_size, random_state=seed, stratify=y
        )
        np.save(str(tr_p), tr_idx)
        np.save(str(te_p), te_idx)
        print(f"  Created + saved split '{tag}'")
    return tr_idx, te_idx

## 1. Download & Load Data

In [ ]:
# Source: Kaggle — Home Credit Default Risk
# https://www.kaggle.com/competitions/home-credit-default-risk
# Requires: ~/.kaggle/kaggle.json  (see data/README.md)
csv_path = DATA_DIR / "application_train.csv"

if not csv_path.exists():
    print("Downloading Home Credit dataset via Kaggle API...")
    subprocess.run(
        [sys.executable, "-m", "kaggle",
         "competitions", "download",
         "-c", "home-credit-default-risk",
         "-p", str(DATA_DIR)],
        check=True,
    )
    for fname in DATA_DIR.iterdir():
        if fname.suffix == ".zip":
            with zipfile.ZipFile(fname, "r") as z:
                # Extract only application_train.csv (anywhere in the archive)
                for member in z.namelist():
                    if "application_train" in member and member.endswith(".csv"):
                        z.extract(member, DATA_DIR)
                        extracted = DATA_DIR / member
                        if extracted != csv_path:
                            extracted.rename(csv_path)
                        break

print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)

# [3] Data provenance — hash first 10k rows for speed
h = _sha256_df(df.iloc[:10000].round(6))
_verify_hash("home_credit_10k", h, ARTIFACTS / "data_hashes.json")
print(f"SHA-256 (first 10k rows): {h[:32]}...")

# Preprocess
target_col = "TARGET"
assert target_col in df.columns, f"Column '{target_col}' not found"
df = df.dropna(subset=[target_col])
y_all   = df[target_col].values.astype(float)
X_df_hc = df.drop(columns=[target_col])

# Label-encode categoricals, median-impute numerics
for col in X_df_hc.select_dtypes(["object", "category"]).columns:
    X_df_hc[col] = X_df_hc[col].astype("category").cat.codes
X_df_hc = X_df_hc.fillna(X_df_hc.median(numeric_only=True))
X_all   = X_df_hc.values.astype(float)

print(f"Samples: {len(y_all):,}  |  Features: {X_all.shape[1]}  |  DR: {y_all.mean():.1%}")
print("  Paper: ~300k accounts, 8.0% default rate")

## 2. Train / Test Split

In [ ]:
# [4] Split indices saved to disk
print("Loading/creating train-test split...")
train_idx, test_idx = _get_or_create_splits(
    X_all, y_all, test_size=0.10, seed=SEED_SPLIT, tag="main", splits_dir=SPLITS_DIR
)
X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print(f"Train DR: {y_train.mean():.1%}  |  Test DR: {y_test.mean():.1%}")

## 3. GBM Baseline

In [ ]:
# [6] Explicit random_state
seed_everything(SEED_GBM)
gbm = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED_GBM,
)
gbm.fit(X_train, y_train)
F_train_gbm = gbm.predict(X_train)
F_test_gbm  = gbm.predict(X_test)

gbm_auc = auc_score(y_test, F_test_gbm)
gbm_ks  = ks_score(y_test, F_test_gbm)
gbm_mse = mse_score(y_test, F_test_gbm)

print(f"GBM       AUC: {gbm_auc:.4f}  KS: {gbm_ks:.4f}  MSE: {gbm_mse:.4f}")
print(f"  Paper:  AUC: 0.7517      KS: 0.3784      MSE: 0.0683")

## 4. QPM on top of GBM

In [ ]:
qpm = QPMQuantizer(
    focus_pd=(0.05, 0.20),
    n_low_max=8, n_mid_max=8, n_high_max=8,
    min_band_size=300,
    pd_cap=0.80,
    monotone_smooth=True,
)
qpm.fit(F_train_gbm, y_train)
F_qpm = qpm.predict(F_test_gbm)

qpm_q_auc = auc_score(y_test, F_qpm)
qpm_q_ks  = ks_score(y_test, F_qpm)
qpm_q_mse = mse_score(y_test, F_qpm)

print(f"K={qpm.n_bins_} bins  "
      f"n_low={qpm.chosen_n_[0]}, n_mid={qpm.chosen_n_[1]}, n_high={qpm.chosen_n_[2]}")
print(f"QPM quant  AUC: {qpm_q_auc:.4f}  KS: {qpm_q_ks:.4f}  MSE: {qpm_q_mse:.4f}")
print(f"  Paper:   AUC: 0.7487      KS: 0.3740      MSE: 0.0683")

## 5. Table 3 — AUC / KS / MSE with 95% BCa CIs

2000 bootstrap resamples (Appendix E of the paper).

In [ ]:
print(f"Computing BCa CIs ({N_BOOT} resamples, seed={SEED_BOOT})...")

gbm_auc_lo, gbm_auc_hi = bootstrap_ci(y_test, F_test_gbm, auc_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)
gbm_ks_lo,  gbm_ks_hi  = bootstrap_ci(y_test, F_test_gbm, ks_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)
qq_auc_lo,  qq_auc_hi  = bootstrap_ci(y_test, F_qpm, auc_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)
qq_ks_lo,   qq_ks_hi   = bootstrap_ci(y_test, F_qpm, ks_score,
                                       n_boot=N_BOOT, method="bca", seed=SEED_BOOT)

print(f"\nTable 3 — Home Credit")
print(f"{'Model':<19} {'MSE':>7} {'AUC':>7} {'95% CI AUC':>17} {'KS':>7} {'95% CI KS':>17}")
print("-" * 78)
print(f"{'GBM':<19} {gbm_mse:.4f}  {gbm_auc:.4f} [{gbm_auc_lo:.3f}-{gbm_auc_hi:.3f}]  "
      f"{gbm_ks:.4f} [{gbm_ks_lo:.3f}-{gbm_ks_hi:.3f}]")
print(f"{'QPM (quantized)':<19} {qpm_q_mse:.4f}  {qpm_q_auc:.4f} [{qq_auc_lo:.3f}-{qq_auc_hi:.3f}]  "
      f"{qpm_q_ks:.4f} [{qq_ks_lo:.3f}-{qq_ks_hi:.3f}]")

## 6. Churn Analysis (Section 4.2)

QPM fine-tuning (fixed edges) vs GBM full retrain on 5 global percentile bands.
Paper reports 3.4x reduction: GBM retrain=0.003635, QPM fine-tune=0.001067.

In [ ]:
# Slice training data into two halves (simulates periodic retraining)
N_HALF = len(y_train) // 2
X_s1, y_s1 = X_train[:N_HALF], y_train[:N_HALF]
X_s2, y_s2 = X_train[N_HALF:2 * N_HALF], y_train[N_HALF:2 * N_HALF]
print(f"Slice1: {len(y_s1):,}  |  Slice2: {len(y_s2):,}")


def _make_gbm():
    return xgb.XGBRegressor(
        n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH, subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE_BYTREE, min_child_weight=MIN_CHILD_WEIGHT,
        gamma=GAMMA, objective="reg:squarederror",
        tree_method="hist", random_state=SEED_GBM,
    )


seed_everything(SEED_GBM)
gbm_v1 = _make_gbm(); gbm_v1.fit(X_s1, y_s1)
seed_everything(SEED_GBM)
gbm_v2 = _make_gbm(); gbm_v2.fit(X_s2, y_s2)

F_v1 = gbm_v1.predict(X_test)
F_v2 = gbm_v2.predict(X_test)

# GBM full retrain: global band churn
gbm_ch = churn_metrics(F_v1, F_v2, n_global_bands=5)
print(f"GBM retrain  global band churn: {gbm_ch['band_churn_rate']:.6f}")
print(f"  Paper: 0.003635")

# QPM fine-tune: fixed v1 edges, v2 GBM scores
F_s1_tr = gbm_v1.predict(X_s1)
qpm_v1  = QPMQuantizer(focus_pd=(0.05, 0.20), n_low_max=8, n_mid_max=8, n_high_max=8,
                        min_band_size=300, pd_cap=0.80)
qpm_v1.fit(F_s1_tr, y_s1)

F_qpm_v1 = qpm_v1.predict(F_v1)
F_qpm_v2 = qpm_v1.predict(F_v2)   # fixed edges applied to new GBM scores
qpm_ch   = churn_metrics(F_qpm_v1, F_qpm_v2, n_global_bands=5)
print(f"QPM fine-tune global band churn: {qpm_ch['band_churn_rate']:.6f}")
print(f"  Paper: 0.001067")

factor_34 = gbm_ch["band_churn_rate"] / max(qpm_ch["band_churn_rate"], 1e-9)
print(f"Reduction factor: {factor_34:.1f}x  (paper: 3.4x)")

In [ ]:
# [7] Results contract — collect + write report
report = {
    "experiment": "02_home_credit",
    "fast_run": FAST_RUN,
    "seeds": {"split": SEED_SPLIT, "gbm": SEED_GBM, "boot": SEED_BOOT},
    "n_estimators": N_ESTIMATORS,
    "n_boot": N_BOOT,
    "gbm": {"auc": gbm_auc, "ks": gbm_ks, "mse": gbm_mse},
    "qpm": {
        "n_bins": qpm.n_bins_,
        "quantized": {"auc": qpm_q_auc, "ks": qpm_q_ks, "mse": qpm_q_mse},
    },
    "churn": {
        "gbm_retrain_global":  gbm_ch["band_churn_rate"],
        "qpm_finetune_global": qpm_ch["band_churn_rate"],
        "reduction_factor":    factor_34,
    },
}

report_path = ARTIFACTS / "report.json"
report_path.write_text(_json.dumps(report, indent=2))
print(f"Report saved: {report_path}")

# [7] Final assertions
assert report["gbm"]["auc"] > 0.72, f"GBM AUC too low: {report['gbm']['auc']:.4f}"
assert report["qpm"]["n_bins"] >= 10, f"Expected >=10 bins, got {report['qpm']['n_bins']}"
assert report["churn"]["reduction_factor"] > 1.5, (
    f"Expected >1.5x reduction, got {report['churn']['reduction_factor']:.1f}x"
)

print("=" * 45)
print("All assertions PASSED")
print(f"  GBM AUC:   {report['gbm']['auc']:.4f}  (paper: 0.7517)")
print(f"  QPM bins:  {report['qpm']['n_bins']}        (paper: varies)")
print(f"  Churn GBM: {report['churn']['gbm_retrain_global']:.6f}  (paper: 0.003635)")
print(f"  Churn QPM: {report['churn']['qpm_finetune_global']:.6f}  (paper: 0.001067)")
print(f"  Factor:    {report['churn']['reduction_factor']:.1f}x  (paper: 3.4x)")